
<font color=#009afa size=50>Blockchain.com API</font>

<font size=50>DAT5002 - Blockchain Fundamentals</font>

|**Assessment Code:** |PRAC1 - Section 2|
|---|---|
|**Author:**|Tom Rowan, ST20285213|
|**Course:**| DAT5002_S2_25|
|**Course Tutor:**| Dr. Ali Shahaab|

## Initialise Notebook

In [ ]:
# Install Requirements Using Pip
%pip install -r requirements.txt

### Import Python Libraries

**NB: This includes my BlockChain API libraries**

In [ ]:
# Standard Libraries
import sys

# Requests
import requests

# Colorama
from colorama import Fore, Style

# Datetime
from datetime import datetime

# Do not print warnings!
import warnings
warnings.filterwarnings('ignore')

# Import My Blockchain Libraries
import blockchainDataAPI.bcda as bcda
import blockchainDataAPI.bcblock as bcblock

### Define helper functions

In [ ]:
# Function to print coloured output
def print_output (str1, str2, delim=':', colour='yellow'):
    if colour.upper() in dir(Fore):
        colour = colour.upper()
    else:
        colour = 'YELLOW'
    print (f'{getattr(Fore, colour)}{str(str1)}{delim}{Style.RESET_ALL} {str(str2)}')

### Check for Internet Connectivity
We need Internet to query the BlockChain API. This notebook will stop execution if Internet access is not available.

In [ ]:
# Coding Syntax Note: _variable_name --> temporary / locally used variable, discarded after use.

# Check for Internet Connectivity
def check_internet (url_list):
    online = False
    for _url in url_list:
        try:
            response = requests.get(_url, timeout=3)
            if response.status_code == 200:
                online = True
                break
        except requests.RequestException:
            output_string = f"Check Internet failed for {_url}"
            print_output ('[FAIL]',output_string, colour='red', delim='')
    return online


_urls_to_check = ['http://connectivitycheck.gstatic.com/generate_204',
                  'https://blockchain.com',
                  'https://blockchain.info',
                  'https://daemonchild.com']

if check_internet (_urls_to_check ):
    print_output ('[INTERNET]','Internet Connectivity Detected. Using Internet resources.',colour='green', delim='')
    online = True

else:
    print ()
    print_output ('[INTERNET]','No Internet Connectivity.', colour='red', delim='')
    sys.exit('Stopping notebook running here.')


## Task 2.1

Firstly, we need to fetch the latest block from the end of the block chain. This is all done using the BlockChain API. I have created a wrapper module for this.

In [ ]:
# Create BC Data API object
bc=bcda.BlockchainComDataAPI()

# Fetch metadata on the latest block to be mined into a Block object
# This is NOT the full block data, and all we really want is the hash
result = bc.latest_block()

# Initialise a Block object
# This will cause the new block object to internally fetch the rest of the data using the API
if "hash" in result.keys():
    the_block = bcblock.Block(result['hash'])
else:
    print_output ('Error, API returned',result, colour='red')
    sys.exit('Stopping notebook running here.')


In [ ]:
result

In [ ]:
print(the_block)
hash = '000000000000000000004f1a1e9e82f7cf0e368a0c601c8d6e0f6fef31585644'

len(hash)

### i) At the time of your writing, what is the height of the most recent block mined and what is the difficulty level? (5%)

The height of the block is returned by the API natively, and the Block object simply returns this:

In [ ]:
print_output ('Height',the_block.height())
print_output ('Mined at',the_block.mined_at())

The number of leading zeroes required for the target hash of a block is an indicator of the difficulty of generating that hash. As the number of zeroes increases, the difficulty also increases.

In [ ]:
print_output ('Hash',the_block.hash())
print_output ('Number of leading zeroes',the_block.leading_zeroes())

There is also a calculation for the difficulty of the block. This function will work for ANY block loaded into a Block object:

In [ ]:
print_output ('Difficulty', the_block.difficulty())

This can be checked and confirmed for the *latest block only* using a call to a different API:
https://blockchain.info/q/getdifficulty

In [ ]:
import requests
result = requests.get (url='https://blockchain.info/q/getdifficulty')
print_output ('Difficulty of latest block',result.content.decode("utf-8"))

### ii) What is the address of the miner of the block ? Can you unmask the identity of the address? (5%)

The Block object is uses a list derived from a public GitHub repository to be found here: https://github.com/bitcoin-data/mining-pools/. Using this data, it is often possible to look up the address of the mining pool. If 'Unknown' is returned, then is it recommended to check https://www.blockchain.com/explorer/blocks/btc?page=1 and click on the latest block. Sometimes, the miner is recorded in the CoinBase message.

In [ ]:
print_output ('Mined by',the_block.mined_by())

### iii) How many transactions does it contain? (5%)

In [ ]:
print_output ('Number of Transactions', the_block.number_tx())

### iv) Which address has the most balance (5%)

We need to use blocks of 50 addresses to balance the URL max length and the number of requests. The API object handles this, so we just throw it a list of addresses.

In [ ]:
the_address_list = the_block.output_addresses()
balances = bc.balance(the_address_list)

In [ ]:
# Dictionary to hold output
max_balance = { 
    'address': 'blah',
    'balance': -1
}

# Loop through the balances loaded from the API to find the highest

for the_address in balances:
    the_balance = balances[the_address]['final_balance']

    if the_balance > max_balance['balance']:
        max_balance['balance'] = the_balance
        max_balance['address'] = the_address
                    
    # We might find a duplicate value, and should call that out
    elif the_balance == max_balance['balance']:
        print ("Duplicate value found: ", max_balance['address'], max_balance['balance'], the_address, the_balance)


# Balance is in Satoshi, so convert to whole Bitcoin
max_balance['btc_val'] = (max_balance['balance']/10e8)

# We can get this value in GBP and USD from this API endpoint: https://blockchain.info/ticker
exchange_rate_GBP = bc.exchange_rate('GBP')
exchange_rate_USD = bc.exchange_rate('USD')

max_balance['gbp'] = round(exchange_rate_GBP * max_balance['btc_val'],2)
max_balance['usd'] = round(exchange_rate_USD * max_balance['btc_val'],2)

# Output Result
output_string = f"{max_balance['btc_val']} btc ({max_balance['balance']} satoshi) [{max_balance['address']}]"
print_output ('Max Balance in block', output_string)

output_string = "£" + "{:,}".format(max_balance['gbp']) + " or $" + "{:,}".format(max_balance['usd'])
print_output ('Value in traditional currencies', output_string)

### v) Which address has most number of transactions. (5%)

In [ ]:
records = bc.multi_address(the_address_list)

In [ ]:
# Dictionary to hold output
max_tx = { 
    'address': 'blah',
    'number_tx': -1
}

# Loop through the addresses loaded from the API to find the highest number of tx's
for record in records['addresses']:

    num_tx = record['n_tx']
    if num_tx > max_tx['number_tx']:
        max_tx['number_tx'] = num_tx
        max_tx['address'] = record['address']

# Output Result
output_string = f"{max_tx['number_tx']} [{max_tx['address']}]"
print_output ('Address with most transactions',output_string)

### vi) When did this address become active (first transaction on the network) (5)?

In [ ]:
# Set the offset to the number of transactions
# The maximum offset supported by the API is 50000 however
# In this case, we will do our best!

if max_tx['number_tx'] > 50000:
    the_offset = 50000
    words = "earliest available (50K tx distant)"    

else:
    # Set the offset so that we get a couple of transactions
    the_offset = max_tx['number_tx']-2
    words = "first recorded"

# Fetch the record for the address, using the offset
# This takes a LONG while for the API to do the search and retrieval
record = bc.single_address(max_tx['address'], offset=the_offset)

In [ ]:
# Find the last transaction in the result
first_tx_index = len(record['txs'])-1
first_tx = record['txs'][first_tx_index]

# Convert to a readable time value
joined_network = datetime.fromtimestamp(first_tx['time']).strftime("%d/%m/%Y %H:%M")

output_string = f"Address {max_tx['address']} {words} transaction at"
print_output (output_string, joined_network)

## Task 2.2 

### Given the information of the above block, how can you retrieve all bitcoin blockchain data up to 1st Jan 2020? Briefly explain your answer. (No code required, just approach or pseudocode is sufficient). (Max 300 words) (5%)

Each block contains two pieces of information that can be used to perform this task.

- Previous Block Hash
- Timestamp

To collect blocks going backwards in time from the most recent block to one produced on 01/01/2020, the following process can be followed:

1) Get a Unix epoch timestamp for the target date (01/01/2020 00:00 => 1601506800)
2) Fetch the most recent block. Store this in a list.
3) Query the last block in the list for a timestamp and the hash of the previous block.
4) Fetch the block referred to by the hash of the previous block. Store this at the end of the list.
5) Check whether the block timestamp is older than the target timestamp.
6) If so, stop. If not, go back to step 3 and repeat.

Looping through this process means that the list grows until the timestamp of the block at the end of the list is older than the target timestamp.

The actual python code to do this using my BlockChain API and Block objects follows. Note that the *date_string* variable should be set to "01/01/2020 00:00" to download the entire dataset as requested, rather than to a relatively recent value as shown in the demonstration code.

At the time of writing on the 29/10/2025 at about 11am, the estimated number of blocks to download would be:

(1761735600 - 1601506800) / 600 = 267,048

If it took 1 second per API query, this would take approximately 74 hours (3 days) to complete.

In [ ]:
sys.exit('Stopping notebook running automatically past here to maintain demo values.')

In [ ]:
# Empty list of blocks
blocks = []

# Create time stamp for target
date_string = "29/10/2025 09:00"
dt = datetime.strptime(date_string, "%d/%m/%Y %H:%M")
target_timestamp = int(dt.timestamp())

# Initialise a Block object
latest_block = bcblock.Block(bc.latest_block()['hash'])

# Add it to the list
blocks.append(latest_block)

# Get the timestamp from the latest block
timestamp = latest_block.timestamp()

# Estimate number of blocks to fetch
est_blocks = int ((timestamp - target_timestamp) / 600)
print ("Estimated blocks to fetch and download: " + str(est_blocks))

counter = 1

print (f"[{counter}] [Fetching] {latest_block.trunc_hash()} - [OK] - {latest_block.mined_at()}")

# Loop back through the previous blocks and collect them into our list of blocks
while timestamp > target_timestamp:
    # increment counter 
    counter += 1

    # Get the last block in the list of blocks
    tail_index = len(blocks)-1
    prev_block_hash = blocks[tail_index].prev_block_hash()

    # Truncate the hash
    trunc_hash = prev_block_hash[:4] + "..." + prev_block_hash[-4:]
    print (f"[{counter}] [Fetching] {trunc_hash}", end='')

    # Fetch the previous block
    prev_block = bcblock.Block(prev_block_hash)
    
    # Add it to the list
    blocks.append(prev_block)
    timestamp = prev_block.timestamp()

    # Print "mined at" time as a statue update
    print (f" - [OK] - {prev_block.mined_at()}")


In [ ]:
# Loop through the blocks to print out some key details as a demonstration
for b in blocks:
    print (b.trunc_hash(), b.height(), b.mined_at(), b.mined_by())